In [ ]:
import yfinance as yf
import numpy as np
import pandas as pd

# Load tickers (stock-only file from previous cell)
tickers = pd.read_csv('congress_tickers_stocks_only.csv')['ticker'].dropna().unique().tolist()
print(f"Tickers to download: {len(tickers)}")

def download_batch(ticker_list, start='2012-01-01', end='2026-02-10'):
    data = yf.download(
        ticker_list,
        start=start,
        end=end,
        auto_adjust=True,   # adjusted close-like prices
        progress=False,
        group_by='ticker'
    )
    return data

# Split into batches of 200 tickers
batch_size = 200
batches = [tickers[i:i+batch_size] for i in range(0, len(tickers), batch_size)]
print(f"Number of batches: {len(batches)}")


In [ ]:
#Ticker mapping and metadata extraction for industry classification

import yfinance as yf
import pandas as pd
import time

# ==============================================================================
# 1. READ TICKERS AND APPLY MAPPING
# ==============================================================================
input_file = "tickers_with_price_data.csv"

try:
    ticker_df = pd.read_csv(input_file)
    raw_tickers = ticker_df.iloc[:, 0].dropna().astype(str).str.strip().str.upper().tolist()
except FileNotFoundError:
    print(f"Error: Could not find '{input_file}'.")
    raw_tickers = [] 

# We use the same mapping from earlier to ensure consistency
TICKER_MAPPING = {
    'FB': 'META',        
    'RTN': 'RTX',        
    'BRK.B': 'BRK-B',    
    'BRK.A': 'BRK-A',
    'BF.B': 'BF-B',
    'CBS': 'PARA',       
    'VIAB': 'PARA',      
    'CELG': 'BMY',       
    'SYMC': 'GEN'        
}

cleaned_tickers = [TICKER_MAPPING.get(t, t) for t in raw_tickers]
unique_tickers = list(set(cleaned_tickers))

print(f"Commencing metadata extraction for {len(unique_tickers)} unique tickers...")

# ==============================================================================
# 2. EXTRACT SECTOR AND INDUSTRY DATA
# ==============================================================================
# We will store the results in a list of dictionaries
classification_data = []
failed_metadata = []

# Using a loop instead of a bulk download because .info must be called per ticker
for i, ticker in enumerate(unique_tickers):
    if (i + 1) % 50 == 0:
        print(f"   Processed {i + 1} / {len(unique_tickers)}...")
        time.sleep(1) # Brief pause to respect rate limits
    
    try:
        # Fetch the ticker object
        stock = yf.Ticker(ticker)
        
        # The .info attribute is a dictionary containing the metadata
        info_dict = stock.info
        
        # Extract the relevant fields, using "Unknown" if the field is missing
        sector = info_dict.get('sector', 'Unknown')
        industry = info_dict.get('industry', 'Unknown')
        
        # If both are 'Unknown', the ticker is likely a broad ETF, ETN, or dead
        if sector == 'Unknown' and industry == 'Unknown':
            # Check if it's a known fund type
            fund_family = info_dict.get('fundFamily', 'Unknown')
            if fund_family != 'Unknown':
                sector = "Broad Market / ETF"
                industry = "Broad Market / ETF"
        
        classification_data.append({
            'Ticker': ticker,
            'Sector': sector,
            'Industry': industry
        })
        
    except Exception as e:
        # If the API fails entirely for this ticker
        failed_metadata.append(ticker)
        classification_data.append({
            'Ticker': ticker,
            'Sector': 'Failed',
            'Industry': 'Failed'
        })

# ==============================================================================
# 3. EXPORT TO CSV FOR JURISDICTION MAPPING
# ==============================================================================
classification_df = pd.DataFrame(classification_data)

output_file = "stock_industry_classifications.csv"
classification_df.to_csv(output_file, index=False)

# Let's see how many unique industries you actually have to map
unique_industries = classification_df[classification_df['Industry'] != 'Unknown']['Industry'].nunique()

print("\n" + "="*70)
print("METADATA EXTRACTION COMPLETE")
print("="*70)
print(f"Successfully processed: {len(classification_data)}")
print(f"Tickers with entirely missing metadata: {len(failed_metadata)}")
print(f"Total Unique Industries to Map: {unique_industries}")
print(f"Data saved to: {output_file}")
print("="*70)

# Display a preview
classification_df.head()


In [ ]:
import yfinance as yf
import pandas as pd
import numpy as np

START_DATE = '2012-01-01'
END_DATE   = '2025-12-31'

# 1) Load stock-only tickers
tickers = (
    pd.read_csv('congress_tickers_stocks_only.csv')['ticker']
      .dropna().unique().tolist()
)
print(f"Total tickers requested (stocks only): {len(tickers)}")

# 2) Split into batches
batch_size = 150   # adjust if you hit rate limits
batches = [tickers[i:i+batch_size] for i in range(0, len(tickers), batch_size)]
print(f"Number of batches: {len(batches)}")

all_prices = []

for i, batch in enumerate(batches, start=1):
    print(f"\nBatch {i}/{len(batches)} | n={len(batch)}")
    try:
        raw = yf.download(
            tickers=batch,
            start=START_DATE,
            end=END_DATE,
            auto_adjust=True,   # adjusted prices
            group_by='ticker',
            progress=False,
            threads=True
        )
    except Exception as e:
        print(f"⚠️ Batch {i} failed with error: {e}")
        continue

    if raw is None or raw.empty:
        print("  -> Empty batch result, skipping.")
        continue

    # 3) Extract a single adjusted price series per ticker
    #    Handle both single- and multi-index columns robustly
    if isinstance(raw.columns, pd.MultiIndex):
        lvl0 = raw.columns.get_level_values(0)
        lvl1 = raw.columns.get_level_values(1)

        if 'Close' in lvl0:
            close = raw.xs('Close', axis=1, level=0)
        elif 'Adj Close' in lvl0:
            close = raw.xs('Adj Close', axis=1, level=0)
        elif 'Close' in lvl1:
            close = raw.xs('Close', axis=1, level=1)
        elif 'Adj Close' in lvl1:
            close = raw.xs('Adj Close', axis=1, level=1)
        else:
            # Fallback: take the first field level as "price"
            first_field = lvl0[0]
            close = raw.xs(first_field, axis=1, level=0)
    else:
        # Single-index columns: assume these are prices already
        close = raw

    # 4) Drop tickers that are all NaN (no history at all)
    before_cols = close.shape[1]
    close = close.dropna(axis=1, how='all')
    after_cols = close.shape[1]
    print(f"  -> Kept {after_cols}/{before_cols} tickers in this batch")

    if after_cols > 0:
        all_prices.append(close)

# 5) Combine all batches
if all_prices:
    prices = pd.concat(all_prices, axis=1)
    # Remove duplicate ticker columns (can happen across batches)
    prices = prices.loc[:, ~prices.columns.duplicated()]
    prices = prices.sort_index()

    # Final cleaning: again drop any completely empty columns (just in case)
    before_cols = prices.shape[1]
    prices = prices.dropna(axis=1, how='all')
    after_cols = prices.shape[1]

    print(f"\nFinal price panel shape: {prices.shape}")
    print(f"Tickers with some price history: {after_cols}")
    print(f"Tickers with no usable data (dropped): {before_cols - after_cols}")

    prices.to_csv('congress_stock_prices_daily.csv')
    print("✅ Saved congress_stock_prices_daily.csv")

    # Optional: export the tickers that survived
    pd.Series(prices.columns, name='ticker').to_csv(
        'tickers_with_price_data.csv', index=False
    )
    print("✅ Saved tickers_with_price_data.csv")
else:
    print("❌ No price data downloaded at all.")


In [ ]:
import yfinance as yf
import pandas as pd
import numpy as np

START_DATE = '2012-01-01'
END_DATE   = '2025-12-31'

# 1) Download S&P 500 (^GSPC)
sp500 = yf.download(
    '^GSPC',
    start=START_DATE,
    end=END_DATE,
    auto_adjust=True,
    progress=False
)

print("Raw ^GSPC shape:", sp500.shape)
print("Raw ^GSPC columns:", sp500.columns)

if sp500 is None or sp500.empty:
    raise ValueError("No data for ^GSPC – check internet or ticker.")

# 2) Extract a price series
if 'Close' in sp500.columns:
    sp_price = sp500['Close']
elif 'Adj Close' in sp500.columns:
    sp_price = sp500['Adj Close']
else:
    sp_price = sp500.iloc[:, 0]

print("sp_price head:")
print(sp_price.head())

# 3) Compute daily log returns
sp_ret = np.log(sp_price / sp_price.shift(1)).dropna()
sp_ret.name = 'R_mkt'

print("sp_ret type:", type(sp_ret))
print("sp_ret head:")
print(sp_ret.head())

# 4) Build DataFrame and save
# If sp_ret is already a Series, this creates a 1-column DataFrame.
sp_df = pd.DataFrame(sp_ret)
sp_df.to_csv('sp500_returns_daily.csv')
print("S&P 500 price series:", sp_price.index.min(), "→", sp_price.index.max())
print("S&P 500 return series shape:", sp_df.shape)

In [ ]:
import os
import requests

# 1) Set your API key (replace with your actual key or use an env variable)
API_KEY = "hbiUbAdgKTHJzszLuM9VYE5jziPSDl5OTyJdPy9f"  # or: os.environ.get("CONGRESS_GOV_API_KEY")
BASE_URL = "https://api.congress.gov/v3"

def cg_get(endpoint, params=None):
    """
    Thin wrapper around Congress.gov API.
    endpoint: 'committee', 'committee/{congress}/{code}', etc.
    params: dict of query params (api_key added automatically).
    """
    if params is None:
        params = {}
    params = dict(params)
    params["api_key"] = API_KEY

    url = f"{BASE_URL}/{endpoint}"
    r = requests.get(url, params=params)
    if r.status_code != 200:
        raise RuntimeError(f"Congress.gov API {r.status_code}: {r.text[:500]}")
    return r.json()


In [ ]:
import json

# Example: list committees for one congress (adjust congress number if needed)
sample_data = cg_get("committee", params={"congress": 118, "limit": 5})

# Pretty-print the top-level keys
print("Top-level keys:", sample_data.keys())

# Inspect the committees container; adapt key names as needed
# Common patterns: 'committees', 'committee', 'data', etc.
print("\nSample JSON (truncated):")
print(json.dumps(sample_data, indent=2)[:3000])


In [ ]:
import json
from time import sleep

def get_committees_for_congress(congress, limit=250):
    """
    Fetch all committees for a given congress with pagination.
    Uses top-level 'committees' list.
    """
    committees = []
    offset = 0

    while True:
        params = {"congress": congress, "limit": limit, "offset": offset}
        data = cg_get("committee", params=params)

        items = data.get("committees", [])
        if not items:
            break

        committees.extend(items)

        pagination = data.get("pagination", {})
        next_url = pagination.get("next")
        if not next_url:
            break

        offset += limit
        sleep(0.1)
    return committees


def fetch_committee_detail(committee_url):
    """
    Fetch a single committee's detailed JSON, given its 'url' field from the list.
    """
    # committee_url is full, like https://api.congress.gov/v3/committee/house/hsgo00?format=json
    # Strip base and query, then call cg_get.
    path = committee_url.split("api.congress.gov/v3/")[-1]
    if "?" in path:
        endpoint, _ = path.split("?", 1)
    else:
        endpoint = path

    data = cg_get(endpoint, params={"format": "json"})
    return data


def extract_members_from_committee_detail(detail_json, congress):
    """
    Flatten a committee detail JSON into rows: BioGuideID × congress × committee.
    Assumes structure: {'committee': {..., 'members': [...]}}.
    """
    rows = []

    committee = detail_json.get("committee", {})
    committee_name = committee.get("name")
    committee_code = committee.get("systemCode")  # e.g. 'hsgo00'
    chamber = committee.get("chamber")

    members = committee.get("members", [])
    # If members is dict with 'member' key, unwrap
    if isinstance(members, dict):
        members = members.get("member", [])

    for m in members:
        bio = m.get("bioguideId")
        if not bio:
            continue
        role = m.get("role")
        side = m.get("side")

        rows.append({
            "BioGuideID": bio,
            "congress": int(congress),
            "committee_code": committee_code,
            "committee_name": committee_name,
            "chamber_committee": chamber,
            "role": role,
            "side": side,
        })
    return rows


In [ ]:
import pandas as pd
import numpy as np

# 1) Load stock price panel
prices = pd.read_csv('congress_stock_prices_daily.csv', index_col=0, parse_dates=True)
print("Prices shape (T × N):", prices.shape)

# 2) Compute log returns per stock
returns = np.log(prices / prices.shift(1))

# 3) Drop first row (all NaN after shift)
returns = returns.iloc[1:, :]

# 4) (Optional) drop columns that are all NaN
before_cols = returns.shape[1]
returns = returns.dropna(axis=1, how='all')
after_cols = returns.shape[1]

print(f"Return panel shape: {returns.shape}")
print(f"Stocks with some return history: {after_cols}, dropped: {before_cols - after_cols}")

returns.to_csv('congress_stock_returns_daily.csv')
print("✅ Saved congress_stock_returns_daily.csv")
print(returns.iloc[:3, :5])

In [ ]:
import time
import pandas as pd
import json
from pathlib import Path

# ------------------------
# Configuration
# ------------------------
START_CONG = 112
END_CONG   = 119
SLEEP_SEC  = 1.0          # ~1 request per second
OUTFILE    = "congress_committee_membership_by_congress.csv"
STATEFILE  = "fetched_committees.json"

# ------------------------
# Load state (for resume)
# ------------------------
if Path(STATEFILE).exists():
    with open(STATEFILE, "r") as f:
        fetched = set(tuple(x) for x in json.load(f))
else:
    fetched = set()

all_rows = []

# If CSV exists, load it so we don't lose progress
if Path(OUTFILE).exists():
    df_existing = pd.read_csv(OUTFILE)
    all_rows = df_existing.to_dict("records")
else:
    df_existing = None

# ------------------------
# Main loop
# ------------------------
for cong in range(START_CONG, END_CONG + 1):
    print(f"\nCongress {cong}")

    committees = get_committees_for_congress(cong, limit=100)
    print(f"  Committees found: {len(committees)}")

    for item in committees:
        # Restrict to standing committees only (correct field name)
        if item.get("committeeTypeCode", "").lower() != "standing":
            continue

        url = item.get("url")
        if not url:
            continue

        key = (cong, url)
        if key in fetched:
            continue

        try:
            detail = fetch_committee_detail(url)
        except Exception as e:
            print(f"⚠️ Error fetching {url}: {e}")
            # back off and continue with next committee
            time.sleep(5)
            continue

        rows = extract_members_from_committee_detail(detail, cong)
        all_rows.extend(rows)

        # Persist progress immediately
        fetched.add(key)
        pd.DataFrame(all_rows).to_csv(OUTFILE, index=False)
        with open(STATEFILE, "w") as f:
            json.dump(list(fetched), f)

        print(f"  ✔ fetched {item.get('name')} ({cong}), members: {len(rows)}")

        time.sleep(SLEEP_SEC)

print("\n✅ Finished safely")
print("Total rows collected:", len(all_rows))


In [ ]:
import pandas as pd
import re
from difflib import SequenceMatcher

# --- load data ---

df_comm = pd.read_csv("member_committees_113_118.csv")
leg = pd.read_csv("MOC_list.csv")  # id, unaccentedGivenName, unaccentedFamilyName
leg = leg.rename(columns={
    "unaccentedGivenName": "first",
    "unaccentedFamilyName": "last",
})

# --- normalisation helpers ---

def norm_name(s):
    if not isinstance(s, str):
        return None
    s = re.sub(r"\s+", " ", s.strip())
    s = s.replace(".", "")      # remove dots
    s = s.upper()
    return s

def split_member_clean(member_raw):
    # 'Last, First Middle ...' -> (Last, First)
    if not isinstance(member_raw, str):
        return None, None
    parts = member_raw.split(",")
    if len(parts) < 2:
        return None, None
    last = parts[0].strip()
    after = parts[1].strip()
    first = after.split()[0].strip()
    return last, first

# committee side keys
df_comm[["last_raw", "first_raw"]] = df_comm["member_raw"].apply(
    lambda x: pd.Series(split_member_clean(x))
)
df_comm["name_key"] = (df_comm["first_raw"] + " " + df_comm["last_raw"]).apply(norm_name)

# bioguide side keys
leg["full_name"] = (leg["first"].fillna("") + " " + leg["last"].fillna("")).str.strip()
leg["name_key"] = leg["full_name"].apply(norm_name)

# --- 1) exact merge ---

df_merged = df_comm.merge(
    leg[["name_key", "id"]],
    on="name_key",
    how="left",
)

print("Exact match rate:", df_merged["id"].notna().mean())

# --- 2) fuzzy fallback for still-unmatched ---

unmatched_mask = df_merged["id"].isna()
unmatched_names = df_merged.loc[unmatched_mask, "name_key"].dropna().unique()
leg_names = leg[["name_key", "id"]].drop_duplicates()

def best_match(name, candidates, threshold=0.95):
    best_id = None
    best_score = 0.0
    for _, row in candidates.iterrows():
        score = SequenceMatcher(None, name, row["name_key"]).ratio()
        if score > best_score:
            best_score = score
            best_id = row["id"]
    if best_score >= threshold:
        return best_id
    return None

fuzzy_map = {}
for name in unmatched_names:
    bid = best_match(name, leg_names, threshold=0.95)
    if bid is not None:
        fuzzy_map[name] = bid

print("Fuzzy matches found:", len(fuzzy_map))

# apply fuzzy matches
df_merged.loc[unmatched_mask, "id"] = df_merged.loc[unmatched_mask, "name_key"].map(fuzzy_map)

print("Total match rate after fuzzy:", df_merged["id"].notna().mean())

# export remaining unmatched for manual inspection
still_unmatched = (
    df_merged[df_merged["id"].isna()][
        ["member_raw", "first_raw", "last_raw", "name_key"]
    ]
    .drop_duplicates()
    .sort_values(["last_raw", "first_raw"])
)
still_unmatched.to_csv("unmatched_committees_manual_fix.csv", index=False)
print("Remaining unmatched unique members:", len(still_unmatched))

# save final file
df_merged.to_csv("member_committees_with_bioguide.csv", index=False)


In [ ]:
import pandas as pd
import yfinance as yf
from time import sleep

# ============================================================
# 0. Load data
# ============================================================

trades = pd.read_csv("Trading_Data.csv")

# Committee data with bioguide ids (your cleaned file)
comm = pd.read_csv("member_committees_with_bioguide_final.csv")
comm = comm[comm["id"].notna()].copy()
comm = comm.rename(columns={"id": "BioGuideID"})

# ============================================================
# 1. Build BioGuideID × committee list (ignoring congress for now)
# ============================================================

member_comm = (
    comm.groupby("BioGuideID")["committee"]
        .apply(list)
        .reset_index()
)

# ============================================================
# 2. Retrieve sector information for all tickers
# ============================================================

trades["Ticker"] = trades["Ticker"].astype(str).str.strip().str.upper()
unique_tickers = trades["Ticker"].dropna().unique()

sector_rows = []
for t in unique_tickers:
    try:
        info = yf.Ticker(t).info
        sector = info.get("sector")
        industry = info.get("industry")
    except Exception:
        sector = None
        industry = None
    sector_rows.append(
        {"Ticker": t, "sector_raw": sector, "industry_raw": industry}
    )
    sleep(0.1)

df_sectors = pd.DataFrame(sector_rows)

# ============================================================
# 3. Map raw sectors to macro sectors
# ============================================================

SECTOR_MAP = {
    "Financial Services": "Financials",
    "Financial": "Financials",
    "Financials": "Financials",
    "Banks": "Financials",
    "Energy": "Energy",
    "Utilities": "Utilities",
    "Industrials": "Industrials",
    "Industrial": "Industrials",
    "Capital Goods": "Industrials",
    "Technology": "Technology",
    "Information Technology": "Technology",
    "Tech": "Technology",
    "Communication Services": "Communications",
    "Telecommunication Services": "Communications",
    "Healthcare": "Healthcare",
    "Health Care": "Healthcare",
    "Consumer Defensive": "Consumer",
    "Consumer Cyclical": "Consumer",
    "Consumer Staples": "Consumer",
    "Consumer Discretionary": "Consumer",
    "Basic Materials": "Materials",
    "Materials": "Materials",
    "Real Estate": "Real Estate",
}

def normalize_sector(sector):
    if sector is None:
        return None
    return SECTOR_MAP.get(sector, None)

df_sectors["sector_macro"] = df_sectors["sector_raw"].apply(normalize_sector)

# ============================================================
# 4. Attach macro sector and committees to each trade
# ============================================================

# Merge sectors
trades_sectors = trades.merge(
    df_sectors[["Ticker", "sector_macro"]],
    on="Ticker",
    how="left"
)

# Merge committee memberships by BioGuideID
trades_sectors = trades_sectors.merge(
    member_comm,
    on="BioGuideID",
    how="left"
)

trades_sectors["committee"] = trades_sectors["committee"].apply(
    lambda x: x if isinstance(x, list) else []
)

# ============================================================
# 5. Define committee–sector relevance and build dummy
# ============================================================

COMMITTEE_SECTORS = {
    "Financial Services": {"Financials"},
    "Ways and Means": {"Financials", "Healthcare", "Consumer"},
    "Energy and Commerce": {"Energy", "Utilities", "Healthcare", "Technology", "Communications"},
    "Natural Resources": {"Energy", "Materials"},
    "Armed Services": {"Industrials"},
    "Transportation and Infrastructure": {"Industrials"},
    "Agriculture": {"Materials", "Consumer"},
    "Homeland Security": {"Industrials", "Communications"},
}

def is_relevant_trade(committees, sector):
    if sector is None:
        return 0
    for c in committees:
        allowed = COMMITTEE_SECTORS.get(c)
        if allowed and sector in allowed:
            return 1
    return 0

trades_sectors["relevant_committee"] = trades_sectors.apply(
    lambda r: is_relevant_trade(r["committee"], r["sector_macro"]),
    axis=1
)

# ============================================================
# 6. Save result WITHOUT touching Trading_Data.csv
# ============================================================

out_path = "Trading_Data_with_sectors_and_committees.csv"
trades_sectors.to_csv(out_path, index=False)
print(f"Saved {len(trades_sectors)} rows to {out_path}")


In [ ]:
import pandas as pd
import requests
import yaml

def fetch_and_parse_yaml(url, target_bioguides):
    print(f"Fetching data from {url}...")
    response = requests.get(url)
    response.raise_for_status()
    data = yaml.safe_load(response.text)
    
    moc_info = []
    party_map = {'Democrat': 'D', 'Democratic': 'D', 'Republican': 'R', 'Independent': 'I', 'Libertarian': 'I'}
    chamber_map = {'rep': 'House', 'sen': 'Senate'}
    
    for leg in data:
        bg_id = leg.get('id', {}).get('bioguide')
        if not bg_id or bg_id not in target_bioguides:
            continue
            
        # Core Identifiers
        gt_id = leg.get('id', {}).get('govtrack')
        icpsr_id = leg.get('id', {}).get('icpsr')
        opensecrets_id = leg.get('id', {}).get('opensecrets')
        
        # Names & Demographics
        first = leg.get('name', {}).get('first', '')
        last = leg.get('name', {}).get('last', '')
        gender = leg.get('bio', {}).get('gender')
        birthday = leg.get('bio', {}).get('birthday')
        
        # Term Data
        terms = leg.get('terms', [])
        if terms:
            first_start = pd.to_datetime(terms[0].get('start'))
            last_end = pd.to_datetime(terms[-1].get('end'))
            years = (last_end - first_start).days / 365.25
            start_str = first_start.strftime('%Y-%m-%d')
            end_str = last_end.strftime('%Y-%m-%d')
            
            # Use the most recent term for Chamber and State
            most_recent_term = terms[-1]
            chamber = chamber_map.get(most_recent_term.get('type'), 'Unknown')
            state = most_recent_term.get('state')
            
            parties = [t.get('party') for t in terms if t.get('party')]
            primary_party_raw = max(set(parties), key=parties.count) if parties else 'Unknown'
            primary = party_map.get(primary_party_raw, primary_party_raw)
        else:
            start_str, end_str, years = None, None, None
            primary, chamber, state = 'Unknown', 'Unknown', None
            
        moc_info.append({
            'BioGuideID': bg_id,
            'GovtrackID': gt_id,
            'ICPSR_ID': icpsr_id,
            'OpenSecretsID': opensecrets_id,
            'UnaccentedFirstName': first,
            'UnaccentedLastName': last,
            'Gender': gender,
            'Birthday': birthday,
            'State': state,
            'Chamber': chamber,
            'Party': primary,
            'TermStart': start_str,
            'TermEnd': end_str,
            'YearsInCongress': round(years, 1) if years else None
        })
        
    return pd.DataFrame(moc_info)

# 1. Load target BioGuide IDs
print("Loading MOC_List.csv...")
moc_list = pd.read_csv('MOC_List.csv')

# Safely find the column containing the BioGuide IDs
bg_col = [c for c in moc_list.columns if 'bioguide' in c.lower() or 'id' in c.lower()][0]
target_bioguides = set(moc_list[bg_col].astype(str).tolist())
print(f"Found {len(target_bioguides)} unique Members of Congress to match.")

# 2. URLs for the official GovTrack files
url_current = "https://raw.githubusercontent.com/unitedstates/congress-legislators/main/legislators-current.yaml"
url_historical = "https://raw.githubusercontent.com/unitedstates/congress-legislators/main/legislators-historical.yaml"

# 3. Fetch and process
df_current = fetch_and_parse_yaml(url_current, target_bioguides)
print(f"Matched {len(df_current)} currently serving members.")

df_historical = fetch_and_parse_yaml(url_historical, target_bioguides)
print(f"Matched {len(df_historical)} historical/former members.")

# 4. Combine, deduplicate, and export
df_combined = pd.concat([df_current, df_historical], ignore_index=True)
df_combined = df_combined.drop_duplicates(subset=['BioGuideID'], keep='first')
df_combined = df_combined.sort_values(['UnaccentedLastName', 'UnaccentedFirstName'])

output_file = 'GovTrack_Master_Enhanced.csv'
df_combined.to_csv(output_file, index=False)
print(f"Success! Exported enhanced dataset with {len(df_combined)} rows to '{output_file}'.")


In [ ]:
# GOVTRACK COMMITTEE ASSIGNMENTS PANEL

import xml.etree.ElementTree as ET
import pandas as pd
import glob
import os

# 1. Load the Master Crosswalk
print("Loading GovTrack/BioGuide crosswalk...")
try:
    master_df = pd.read_csv('GovTrack_Master_Enhanced.csv')
    master_df['GovtrackID'] = master_df['GovtrackID'].astype(str).str.replace('.0', '', regex=False)
    id_map = dict(zip(master_df['GovtrackID'], master_df['BioGuideID']))
    print(f"Loaded {len(id_map)} ID mappings.")
except FileNotFoundError:
    print("Error: 'GovTrack_Master_Enhanced.csv' not found. Ensure it is in the same folder.")
    id_map = {}

# 2. Function to parse XML and separate Parent/Subcommittee columns
def parse_committee_xml(file_path, session_number, id_mapping):
    tree = ET.parse(file_path)
    root = tree.getroot()
    
    records = []
    
    for comm in root.findall('.//committee'):
        parent_code = comm.get('code')
        # Clean parent name
        parent_name = comm.get('displayname', '').strip()
        comm_type = comm.get('type')
        
        # Parse main committee members
        for member in comm.findall('member'):
            govtrack_id = member.get('id')
            role = member.get('role') if member.get('role') else 'Member'
            bioguide_id = id_mapping.get(govtrack_id, "UNKNOWN")
            
            records.append({
                'BioGuideID': bioguide_id,
                'GovTrackID': govtrack_id,
                'Congress_Session': session_number,
                'Committee_Code': parent_code,
                'Parent_Committee_Name': parent_name,
                'Subcommittee_Name': None,  # No subcommittee for parent-level membership
                'Chamber': comm_type,
                'Role': role,
                'Is_Subcommittee': 0
            })
            
        # Parse subcommittee members
        for subcomm in comm.findall('.//subcommittee'):
            sub_code = subcomm.get('code')
            # Clean subcommittee name
            sub_name = subcomm.get('displayname', '').strip()
            
            for sub_member in subcomm.findall('member'):
                sub_govtrack_id = sub_member.get('id')
                sub_role = sub_member.get('role') if sub_member.get('role') else 'Member'
                sub_bioguide_id = id_mapping.get(sub_govtrack_id, "UNKNOWN")
                
                records.append({
                    'BioGuideID': sub_bioguide_id,
                    'GovTrackID': sub_govtrack_id,
                    'Congress_Session': session_number,
                    'Committee_Code': f"{parent_code}-{sub_code}",
                    'Parent_Committee_Name': parent_name,     # Retain parent name
                    'Subcommittee_Name': sub_name,            # Isolate subcommittee name
                    'Chamber': comm_type,
                    'Role': sub_role,
                    'Is_Subcommittee': 1
                })
                
    return pd.DataFrame(records)

# 3. Loop through all session XMLs
xml_files = glob.glob('[0-9]*.xml')
all_data = []

for file in xml_files:
    session_num = int(os.path.basename(file).replace('.xml', ''))
    print(f"Parsing Session {session_num}...")
    
    df_session = parse_committee_xml(file, session_num, id_map)
    all_data.append(df_session)

# 4. Combine and Export
if all_data:
    final_panel = pd.concat(all_data, ignore_index=True)
    
    # Reorder columns logically
    cols = ['BioGuideID', 'GovTrackID', 'Congress_Session', 'Chamber', 
            'Committee_Code', 'Parent_Committee_Name', 'Subcommittee_Name', 
            'Role', 'Is_Subcommittee']
    final_panel = final_panel[cols]
    
    # Sort for an organized panel output
    final_panel = final_panel.sort_values(by=['Congress_Session', 'Parent_Committee_Name', 'Is_Subcommittee', 'BioGuideID'])
    
    final_panel.to_csv('Committee_Assignments_Panel.csv', index=False)
    
    print(f"\nSuccess! Extracted {len(final_panel)} committee assignments into a clean panel.")
else:
    print("No XML files found to process.")


In [ ]:
# Find the most active committees by jurisdictional trades

import pandas as pd

# 1. Load the files
trades = pd.read_csv('Trading_Data_with_Jurisdiction.csv')
assignments = pd.read_csv('Committee_Assignments_Panel.csv')
matrix = pd.read_csv('Jurisdictional_Matrix_Final.csv')

# 2. Prepare the Matrix (Exploding Mapped_Industries)
matrix.columns = matrix.columns.str.strip()

# Note the plural 'Mapped_Industries' to match your matrix format
matrix['ind_list'] = matrix['Mapped_Industries'].str.split(';')
matrix_exploded = matrix.explode('ind_list')
matrix_exploded['ind_list'] = matrix_exploded['ind_list'].str.strip()

# 3. Align Trades and Sessions (2013-2025)
trades['Traded'] = pd.to_datetime(trades['Traded'])
trades = trades[(trades['Traded'] >= '2013-01-01') & (trades['Traded'] <= '2025-12-31')].copy()
trades = trades[~trades['Transaction'].str.contains('Exchange', case=False, na=False)].copy()

def get_congress(dt):
    year = dt.year
    if year < 2013: return 112
    return 112 + (year - 2011) // 2

trades['Congress_Session'] = trades['Traded'].apply(get_congress)

# 4. Join Trades to Assignments (Get Committee Codes for each member)
trade_assignments = trades.merge(
    assignments[['BioGuideID', 'Congress_Session', 'Committee_Code']], 
    on=['BioGuideID', 'Congress_Session'], 
    how='inner'
)

# 5. Join with the Exploded Matrix (Matching on Industry)
# We match the trade's 'Industry' to the matrix's 'ind_list'
full_map = trade_assignments.merge(
    matrix_exploded[['Committee_Code', 'Parent_Committee_Name', 'ind_list']], 
    left_on=['Committee_Code', 'Industry'],
    right_on=['Committee_Code', 'ind_list'],
    how='inner'
)

# 6. Tally the Jurisdictional Trades
committee_counts = (
    full_map.groupby('Parent_Committee_Name')
    .size()
    .reset_index(name='Jurisdictional_N')
    .sort_values('Jurisdictional_N', ascending=False)
)

print("=== JURISDICTIONAL TRADES BY COMMITTEE (INDUSTRY-BASED) ===")
print(committee_counts)

# Save for your results section
committee_counts.to_csv('N_per_Committee_Industry.csv', index=False)